# Evaluación adversarial del modelo v3 con IBM ART

**Objetivo:** cuantificar la robustez del Random Forest v3 frente a ataques de evasión black-box. El modelo se entrena asumiendo que el atacante envía features "honestas"; aquí medimos qué ocurre cuando el atacante perturba las features para eludir la detección.

**Método:**
1. Cargar el RF v3 desde `/home/app/models/` (ya verificado por hash en ml_api).
2. Tomar un subset de 500 muestras de ataque del test set de CICIDS (features + labels).
3. Aplicar ataque **ZOO** (Zeroth-Order Optimization — Chen et al. 2017) — black-box, solo usa queries al modelo, realista para un atacante de red.
4. Reportar **tasa de evasión** = % de muestras originalmente clasificadas como ataque que tras perturbación se clasifican como Benign.

**Referencias:**
- Chen, P.Y. et al. (2017). *ZOO: Zeroth Order Optimization based Black-box Attacks to Deep Neural Networks*.
- Apruzzese, G. et al. (2022). *Modeling realistic adversarial attacks against NIDS*. ACM DTRAP.
- IBM ART docs: https://adversarial-robustness-toolbox.readthedocs.io

In [ ]:
import json
import hashlib
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from art.estimators.classification import SklearnClassifier
from art.attacks.evasion import ZooAttack, HopSkipJump

MODEL_DIR = Path('/home/app/models')
DATASETS = Path('/home/app/datasets')
print('Modelos:', sorted([p.name for p in MODEL_DIR.iterdir()]))

## 1. Verificar integridad + cargar v3

In [ ]:
def sha256(p):
    h = hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

manifest = json.loads((MODEL_DIR / 'manifest.json').read_text())
for name, expected in manifest.items():
    actual = sha256(MODEL_DIR / name)
    status = '✓' if actual == expected else '✗ TAMPERED'
    print(f'  {status}  {name}')

rf = joblib.load(MODEL_DIR / 'rf_multiclass_v3.joblib')
scaler = joblib.load(MODEL_DIR / 'scaler_v3.joblib')
le = joblib.load(MODEL_DIR / 'label_encoder_v3.joblib')
feature_names = joblib.load(MODEL_DIR / 'feature_names_v3.joblib')
print(f'Modelo v3 cargado: {len(feature_names)} features, clases={list(le.classes_)}')

## 2. Seleccionar muestras de ataque (500 flujos)

Usamos un subset pequeño del dataset CICIDS ya preprocesado. Para reproducibilidad se puede regenerar con `scripts/09_train_v3.py`.

Si el dataset no está disponible, creamos muestras sintéticas desde patrones aprendidos por el modelo (sampling por clase).

In [ ]:
# Intentamos cargar test set real; si no, generamos representantes sintéticos
test_data_path = DATASETS / 'cicids_v3_test.parquet'

if test_data_path.exists():
    df = pd.read_parquet(test_data_path)
    X_te = df[feature_names].to_numpy()
    y_te = le.transform(df['Label_6'].values)
    print(f'Test set real: {X_te.shape}')
else:
    # Fallback: generar vectores aleatorios y filtrar por los que el modelo
    # clasifica como ataque con alta confianza
    rng = np.random.default_rng(42)
    n_candidates = 5000
    X_rand = rng.uniform(-2, 2, size=(n_candidates, len(feature_names)))
    preds = rf.predict(X_rand)
    # Tomar los que NO son Benign
    benign_idx = list(le.classes_).index('Benign')
    mask = preds != benign_idx
    X_te = X_rand[mask]
    y_te = preds[mask]
    print(f'Fallback sintético: {X_te.shape} vectores clasificados como ataque')

# Seleccionar 500 de ataque
benign_cls = list(le.classes_).index('Benign')
attack_mask = y_te != benign_cls
idx = np.where(attack_mask)[0][:500]
X_sample = X_te[idx]
y_sample = y_te[idx]
print(f'Muestras de ataque para atacar: {X_sample.shape}')
print(f'Distribución de clases: {pd.Series(le.inverse_transform(y_sample)).value_counts().to_dict()}')

## 3. Predicción base (antes del ataque)

In [ ]:
y_pred_orig = rf.predict(X_sample)
acc_orig = np.mean(y_pred_orig == y_sample)
print(f'Accuracy del modelo sobre muestras de ataque ORIGINALES: {acc_orig:.4f}')
print(f'  → {int(acc_orig * len(y_sample))}/{len(y_sample)} clasificadas correctamente')

n_detected = np.sum(y_pred_orig != benign_cls)
print(f'  → {n_detected}/{len(y_sample)} detectadas como ataque (cualquier clase no-Benign)')

## 4. Ataque ZOO (black-box)

ZOO realiza optimización de orden cero — solo necesita queries al modelo (no gradientes). Simula un atacante que solo puede observar las respuestas de la API `/predict`.

**Parámetros:** `max_iter=20` (para completar en tiempo razonable), `confidence=0.01`.

In [ ]:
clf = SklearnClassifier(model=rf, clip_values=(float(X_sample.min()), float(X_sample.max())))

# HopSkipJump — black-box, basado en decision boundary (Chen, Jordan & Wainwright 2020).
# Mas robusto que ZOO para Random Forest (no requiere gradientes, solo labels).
attack = HopSkipJump(
    classifier=clf,
    targeted=False,
    norm=2,
    max_iter=10,
    max_eval=500,
    init_eval=50,
    init_size=50,
    verbose=False,
)

SUBSET = 50  # HopSkipJump tarda ~3-8s por muestra; 50 => 3-7 min total
print(f'Generando ejemplos adversariales con HopSkipJump sobre {SUBSET} muestras...')
X_adv = attack.generate(X_sample[:SUBSET])
print(f'Adversarios generados: {X_adv.shape}')


## 5. Tasa de evasión

In [ ]:
y_adv = rf.predict(X_adv)
y_orig_sub = rf.predict(X_sample[:len(X_adv)])

# Evasion: muestra originalmente detectada como ataque, ahora clasificada como benigno
evaded = (y_orig_sub != benign_cls) & (y_adv == benign_cls)
evasion_rate = evaded.mean()
print(f'Muestras que evadieron deteccion: {evaded.sum()}/{len(y_adv)}')
print(f'TASA DE EVASION: {evasion_rate:.2%}')

perturbation = float(np.linalg.norm(X_adv - X_sample[:len(X_adv)], axis=1).mean())
print(f'Perturbacion L2 promedio: {perturbation:.4f}')


## 6. Visualizar qué features se perturban más

In [ ]:
delta = X_adv - X_sample[:len(X_adv)]
feat_impact = np.abs(delta).mean(axis=0)
imp_df = pd.DataFrame({'feature': feature_names, 'mean_abs_delta': feat_impact})
top = imp_df.sort_values('mean_abs_delta', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top['feature'][::-1], top['mean_abs_delta'][::-1], color='#B85450')
ax.set_xlabel('Perturbacion media absoluta (features escaladas)')
ax.set_title('Features mas modificadas por HopSkipJump')
plt.tight_layout()
plt.savefig('/home/app/logs/art_features_perturbed.png', dpi=110)
plt.close()

print('Top 5 features objetivo de perturbacion:')
for _, r in top.head(5).iterrows():
    print(f'  {r["feature"]:35s}  d={r["mean_abs_delta"]:.4f}')


## 7. Guardar resultados

In [ ]:
results = {
    'attack': 'HopSkipJump (black-box, L2)',
    'model': 'v3',
    'n_samples_attacked': int(len(X_adv)),
    'accuracy_original': float(acc_orig),
    'n_evaded': int(evaded.sum()),
    'evasion_rate': float(evasion_rate),
    'mean_L2_perturbation': float(perturbation),
    'top_perturbed_features': top.head(5).to_dict(orient='records'),
}

out = Path('/home/app/logs/art_evasion_results.json')
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(results, indent=2, ensure_ascii=False))
print(f'Resultados guardados en {out}')
print(json.dumps(results, indent=2, ensure_ascii=False))


## 8. Interpretación para el capstone

- **Tasa de evasión** = vulnerabilidad del modelo ante un atacante que conoce la API (black-box).
- **Perturbación L2** = magnitud del cambio requerido. Si es baja, el atacante gasta poco esfuerzo.
- **Top features perturbadas** = indica dónde refuerza el atacante su ataque — features que el estudiante puede investigar como *defensa*.

**Recomendación de criterio §5.4:** `Tasa de evasión ZOO sobre muestras de ataque ≤ 40 %` (umbral tentativo, afinar tras experimentos completos).

**Limitaciones:**
- ZOO asume que el atacante puede perturbar *cualquier* feature. En la práctica, no se puede cambiar `Destination Port` arbitrariamente sin romper el ataque.
- Para realismo: reducir a perturbaciones solo en features *controllables por el atacante* (padding de paquetes, timing inter-paquetes).